In [ ]:
import pandas as pd
import json

# 1. Carregar Dados Nível 2
with open("../dados/dados_nivel_2.json", encoding="utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame(raw["operacoes"])
taxa_cambio = raw["taxa_cambio_usd_brl"]

# 2. Limpeza e Tratamento
# Remove duplicatas exatas mantendo a primeira ocorrência
df = df.drop_duplicates(subset="id", keep="first").reset_index(drop=True)
# Converte datas (operacoes sem data viram NaT)
df["data_dt"] = pd.to_datetime(df["data"], errors="coerce")
# Normaliza moedas para BRL
df["valor_brl"] = df.apply(lambda r: r["valor"] * taxa_cambio if r["moeda"] == "USD" else r["valor"], axis=1)

# 3. Regra 1: Fracionamento (Soma > 50k, >=3 ops, todas < 20k no mesmo dia)
def aplica_regra_fracionamento(df):
    flagged = set()
    df_com_data = df.dropna(subset=["data_dt"])
    for (cliente, data), grupo in df_com_data.groupby(["cliente_id", "data_dt"]):
        if len(grupo) >= 3 and grupo["valor_brl"].sum() > 50_000 and (grupo["valor_brl"] < 20_000).all():
            flagged.add((cliente, data))
    return flagged

chaves_fracionamento = aplica_regra_fracionamento(df)
df["flag_fracionamento"] = df.apply(
    lambda r: pd.notna(r["data_dt"]) and (r["cliente_id"], r["data_dt"]) in chaves_fracionamento, axis=1
)

# 4. Regra 2: Valor Atípico (Valor > 5x a mediana do cliente, mín. 4 operações)
def aplica_regra_valor_atipico(df):
    flags = pd.Series(False, index=df.index)
    for cliente, grupo in df.groupby("cliente_id"):
        if len(grupo) >= 4:
            mediana = grupo["valor_brl"].median()
            limite = 5 * mediana
            flags.loc[grupo.index] = grupo["valor_brl"] > limite
    return flags

df["flag_valor_atipico"] = aplica_regra_valor_atipico(df)

# 5. Filtrar apenas os clientes suspeitos
clientes_suspeitos = df[df["flag_fracionamento"] | df["flag_valor_atipico"]]["cliente_id"].unique()
print(f"Total de clientes suspeitos encontrados: {len(clientes_suspeitos)}")